# NLP

Es una área de estudio que se enfoca en el procesamiento de la información contenida en lenguaje natural.


## Text Pre Processing

1. Segmentacion de textos y tokenizacion de palabras
2. Identificación del Idioma
3. Eliminar las punctuaciones, digitos, y palabras como articulos, determinantes.


# Tokenización

### Instalación y Configuración Spacy

In [ ]:
!pip install --upgrade spacy
!pip install transformers
!python -m spacy download es_core_news_lg
import spacy
from spacy import displacy
nlp = spacy.load("es_core_news_lg")

In [ ]:
text="""El presidente de la República Pedro Castillo anunció la creación del denominado \
Servicio Civil Agrario - Secigra, con la finalidad de que “miles de jóvenes universitarios \
recién egresados” vayan al campo para brindar apoyo técnico “a nuestros agricultores \
y agricultoras”. “Tenemos en preparación un programa de servicio civil agrario, al que llamamos \
Secigra-Agrario, por lo cual miles de jóvenes universitarios, recién egresados, saldrán al \
campo a apoyar técnicamente a nuestro agricultores”, expresó."""
text

### Tokenización por sentencia

In [ ]:
doc = nlp(text)
for idx,sent in enumerate(doc.sents):
  print(f'sentencia {idx+1}: ', sent)
  print()

### 1) Tokenización por espacio

In [ ]:
print(text.split(' '))

### 2) Tokenización basado en palabras

In [ ]:
from spacy.tokenizer import Tokenizer
from spacy.lang.es import Spanish
nlp = Spanish()
tokens = nlp.tokenizer(text)
print(list(tokens))

### 3) Tokenización para sub-palabras

In [ ]:
from transformers import BertTokenizer
#tz = BertTokenizer.from_pretrained("bert-base-cased")
tz = BertTokenizer.from_pretrained("bert-base-multilingual-cased")

In [ ]:
tz.tokenize(text)

## Feature Encoding - Vectorización

Las oraciones podemos generarlas como vectores. Luego, cada palabra ahora representa un vector, pero esta caracterizado por su contexto en las oraciones que se encuentran. De esta manera, hallamos similaridades entre palabras utilizando vectores.


In [ ]:
!pip install gensim==3.8.0
!pip install pyemd

In [ ]:
!pip install stanza
%matplotlib inline
import glob
import numpy as np
import matplotlib.pyplot as plt
import nltk
from nltk.corpus import stopwords
import csv
import gensim
import pandas as pd
from itertools import groupby
from gensim.similarities import WmdSimilarity
from gensim.models import Word2Vec

nltk.download('punkt') 
nltk.download('stopwords')
stop_words = stopwords.words('spanish')
import stanza
import re
stanza.download('es')
print(stop_words)
nlp = stanza.Pipeline(processors='tokenize',lang='es',use_gpu=True)

In [ ]:
# Quitamos los stop words
# articulos, adverbios
def preprocessing(words):
  clean_sentence=[]
  for word in words:
    word = word.lower()
    if (word not in stop_words) and word.isalpha():
      clean_sentence.append(word)
  return clean_sentence
      
# Tomamos oraciones
# tokenizamos
# Generamos listas de oraciones limpias
def preprocessing_sentences(all_news):
  all_sentences = []
  raw_sentences=[]
  for new in tqdm(all_news):
    doc = nlp(new)
    for sentence in doc.sentences:
      words = [word.text for word in sentence.words]
      if len(words)>5:
        clean_sentence=preprocessing(words)
        all_sentences.append(clean_sentence)
        raw_sentences.append(words)
  return all_sentences,raw_sentences

In [ ]:
# Importar base de datos
df = pd.read_csv( r"https://www.dropbox.com/s/fjiwa26yjobrtfz/sample_news.csv?dl=1")
# Reemplazar np.nan
# por strings vacios
df.body = df.body.replace( np.nan, "")
# String de oraciones
all_news = df.body.str.strip()

In [ ]:
# Procesamiento de las oraciones
from tqdm import tqdm

sentences,raw_sentences = preprocessing_sentences(all_news)
len(sentences)
lens = [len(sentence) for sentence in sentences]
avg_len = sum(lens) / float(len(lens))


In [ ]:
# Plot de Longitud de las oraciones
plt.figure(figsize=(10,6))
plt.hist([len(sentence) for sentence in sentences])
plt.axvline(avg_len, color='#e41a1c')
plt.title('Histograma de longitud de frases.')
plt.xlabel('Longitud')
plt.text(10, 800, 'mean = %.2f' % avg_len)
plt.show()

## Buscando Similaridad
Con la base de datos que tenemos, generamos vectores de representación para cada palabra. De esta manera, podemos encontrar la similaridad entre palabras o frases.

In [ ]:
# Treinamos Word2Vec com todos las frases
model = Word2Vec(sentences, workers=3, sg=1, min_count=3, window=10)
word_vectors = model.wv
# Estoy buscando similaridad
# Los 10 mejores
num_best = 10
# Genero con el modelo utilizando las 10mil primeras oraciones
# Con Insantance genero un buscador de similaridad
instance = WmdSimilarity(sentences[:10000], model, num_best=num_best)

In [ ]:
# Busco alcalde
# proveedores
text = 'alcalde'
sentence = nltk.word_tokenize(text.lower(), language='spanish')
print(sentence)
query = preprocessing(sentence)
print(query)
sims = instance[query]  # A query observa na classe de similaridade.

In [ ]:
# Mostramos los resultados de la pregunta
print('Query:')
print(text)
for i in range(num_best):
    print("")
    print('sim = %.4f' % sims[i][1])
    print(" ".join(sentences[sims[i][0]]))

In [ ]:
model.wv.most_similar('proveedores')

## Reentrenando modelos de NLP (FINE-TUNING)

Grandes empresas y centros de investigación ya generaron modelos pre entrenados con grandes cargas de bases de datos para clasificación, modelos de preguntas y respuestas, etc.

En este caso usaremos BERT (Google AI Language) un modelo pre entrenado que entiendo el contexto del lenguaje. A este modelo, se le necesita hacer un fine-tuning, es decir, reentrenarlo para una tarea en específico que deseamos realizar. En este caso, deseamos clasificar noticias si son o no relevantes. Para esto necesitamos una base de datos ya etiquetada,noticias que previamente ya hemos identificado si son o no relevantes. Y ajustaremos el modelo Bert para la tarea que desamos realizar.



In [ ]:
#install the required libraries
!pip install transformers
!pip install datasets
!pip install pandas
!pip install scikit-learn

In [ ]:
#import what we need later
import datasets
from datasets import load_dataset
from datasets import Dataset, DatasetDict

import pandas as pd

from sklearn.model_selection import train_test_split


In [ ]:
# Importamos la data
our_data = pd.read_csv("https://www.dropbox.com/s/tbjr8g1hslb8q4o/Full-Economic-News-DFE-839861.csv?dl=1" , encoding = "ISO-8859-1" ) \
            .sample( n = 2000 ) \
            .reset_index( drop = True )

In [ ]:
our_data.shape

In [ ]:
# Tomamos solamente dos columnas el texto y Y, relevance
# Cambiamos yes y No por 1 y 0
mylen = len(our_data["text"].tolist())
mytexts = [] #will contain the text strings
mylabels = [] #will contain the label as 1 or 0 (Yes or No respectively)
for i in range(0,mylen):
    if str(our_data['relevance'][i]) == 'yes':
        mytexts.append(str(our_data["text"][i]))
        mylabels.append(1)
    elif str(our_data["relevance"][i]) == "no":
        mytexts.append(str(our_data["text"][i]))
        mylabels.append(0)
    else:
        print("skipping")
len(mytexts)
len(mylabels)

In [ ]:
# Separamos train y test data 
# Solo 25% de data para test
train_texts, test_texts, train_labels, test_labels = train_test_split(mytexts, mylabels, test_size=.25)
train_texts, val_texts, train_labels, val_labels = train_test_split(train_texts, train_labels, test_size=0.1)


In [ ]:
# Obtenemos el modelo bert
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')

# Estemodelo tiene un tokenizador que usaremos para incorporar nuestra
# data
train_encodings = tokenizer(train_texts, truncation=True, padding=True)
val_encodings = tokenizer(val_texts, truncation=True, padding=True)
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [ ]:
# Introducimos nuestra data al modelo
import torch

class MyDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):

        # Seleccionamos los encodings. Texto/data
        # Labels, Y, lo que queremos predesir
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        # Cada texto lo convertimos en un vector tipo tensor
        # De igual manera los labels
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [ ]:
# Generamos la dataset para el input
train_dataset = MyDataset(train_encodings, train_labels)
test_dataset = MyDataset(test_encodings, test_labels)
val_dataset = MyDataset(val_encodings, val_labels)

In [ ]:
# Tomamos Bert para la clasificación
from transformers import BertForSequenceClassification, Trainer, TrainingArguments
from datasets import load_metric

In [ ]:
# Especificamos los argumentos para el re entrenamiento
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # total number of training epochs # number of times to change the model
    per_device_train_batch_size=16,  # batch size per device during training # number of observations
    per_device_eval_batch_size=64,   # batch size for evaluation # of obs for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

# Definimos la metrica para el re entrenamiento
def compute_metrics(eval_preds):
    metric = load_metric("accuracy", "f1")
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Tomamos el modelo
model = BertForSequenceClassification.from_pretrained("bert-base-cased")

# Instance del Re entrenamiento del Modelo
trainer = Trainer(
    model=model,                         # the instantiated 🤗 Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=train_dataset,         # training dataset
    eval_dataset=val_dataset,           # evaluation dataset
    compute_metrics=compute_metrics      #specify metrics

)


In [ ]:
# Entrenando el modelo
trainer.train()

In [ ]:
import numpy as np
# Testeando la predicción del modelo
predictions = trainer.predict(test_dataset)
print(predictions.predictions.shape, predictions.label_ids.shape)
preds = np.argmax(predictions.predictions, axis=-1)
metric =load_metric('accuracy', 'f1')
print(metric.compute(predictions=preds, references=predictions.label_ids))

In [ ]:
# Evaluamos el resultado
from sklearn.metrics import confusion_matrix
print(confusion_matrix(predictions.label_ids, preds, labels=[1,0]))

## References
1. https://econnlpcourse.github.io/
2. https://huggingface.co/docs/transformers/training
3. https://www.dropbox.com/s/6vs85d2lqx03r0e/Abad_Gago_et_al.pdf?dl=0 (GOLAZO)
4. Curso de NLP QLAB